# Phase 1: Data Engineering - Loading & Cleaning
## Project: End-to-End Business Intelligence System
**Role:** Data Engineering Lead
**Objective:** Process the raw `Sample - Superstore.csv` dataset, clean anomalies, handle missing values, and prepare it for feature engineering/database loading.

In [ ]:
import pandas as pd
import numpy as np

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### Step 1: Data Loading & Initial Inspection
Loading the raw data to understand its shape, columns, and variable types.

In [ ]:
# Define the path to our raw dataset
file_path = 'Sample - Superstore.csv'

# Load the CSV using Pandas
# Note: Explicitly setting encoding for standard superstore datasets to avoid reading errors
try:
    df = pd.read_csv(file_path, encoding='windows-1252')
except Exception as e:
    df = pd.read_csv(file_path, encoding='utf-8')

# Quick sanity check of the top 5 rows
df.head()

In [ ]:
# Inspect data types and missing values footprint
df.info()

In [ ]:
# Produce summary statistics for numerical columns
df.describe()

### Step 2: Data Cleaning
Here we execute structured cleaning procedures:
1. **Uniform Headers:** Converting column names to `snake_case` for PostgreSQL friendliness.
2. **Handling Duplicates:** Checking and dropping identical rows.
3. **Missing Values:** Identifying and addressing null values contextually.
4. **Data Type Casting:** Standardizing date fields to `datetime` objects.

In [ ]:
# 2.1 Standardize Column Names (e.g., 'Order Date' -> 'order_date')
# PostgreSQL prefers lowercase column names with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

print("Standardized Columns:")
print(df.columns.tolist())

In [ ]:
# 2.2 Remove Duplicate Rows
initial_shape = df.shape
df = df.drop_duplicates()
final_shape = df.shape

print(f"Dropped {initial_shape[0] - final_shape[0]} duplicate rows")

In [ ]:
# 2.3 Handling Missing Values
print("Missing values per column before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Strategy: 
# Fill missing float/int metrics (like profit/discount) with 0
# Fill missing text variables (like Region/State) with 'Unknown'
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        df[col] = df[col].fillna(0)

print("\nAll missing values addressed!")

In [ ]:
# 2.4 Data Type Casting (Dates)
date_columns = ['order_date', 'ship_date']

for col in date_columns:
    if col in df.columns:
        # Convert to datetime object (Day/Month/Year or Month/Day/Year inferred automatically usually, but adjust format if necessary)
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Sanity check on the newly assigned datatypes
print("Data Types After Conversion:")
print(df[date_columns].dtypes if any(c in df.columns for c in date_columns) else "No date columns found")

In [ ]:
# Validate final clean dataset
print(f"\nDataset ready to move to Phase 3 (Feature Engineering). Final shape: {df.shape}")
df.head()